In [428]:
import os
import enum

from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional, Any

import torch
from torch import Tensor
from torch import quasirandom as t_qrand

import botorch
from botorch import models as b_models
from botorch import test_functions as b_tfuncs
from botorch import acquisition as b_acq
from botorch import fit as b_fit
from botorch import optim as b_optim
from botorch.posteriors import gpytorch as bp_gpytorch
from botorch.utils import transforms as b_transf

import gpytorch
from gpytorch import constraints as g_constraints
from gpytorch import likelihoods as g_lh
from gpytorch import mlls as g_mlls
from gpytorch import kernels as g_kern




device: torch.device = torch.device("mps" if torch.mps.is_available() else "cpu")
tensor_dtype: torch.dtype = torch.float32
runs: int = os.environ.get("SMOKE_TEST", 30)

dim: int = 25
num_samples: int = 5

lb: float = -50
ub: float = 50

confidence: float = 2.5

batch_size: int = 5
max_iterations: int = 30
max_cholesky_size: int = float("inf")


In [429]:
@dataclass
class BOState():
    max_iterations: int
    current_iteration: int = 1

    num_restarts: int = 10
    raw_samples: int = 512

    def is_exceeded(self):
        return self.current_iteration > self.max_iterations
    def increment(self):
        print(f"Current iteration: {self.current_iteration}")
        self.current_iteration = self.current_iteration + 1

In [430]:
ackley: b_tfuncs.Ackley = b_tfuncs.Ackley(
    dim=dim,
    negate=False,
    ).to(
        device=device, 
        dtype=tensor_dtype
        )
ackley.bounds[0, :].fill_(lb)
ackley.bounds[1, :].fill_(ub)

def eval_objective(
        x: Tensor
        ):
    return ackley(b_transf.unnormalize(x, ackley.bounds))

In [431]:
def create_surrogate(
        X: Tensor,
        Y: Tensor,
        noise_interval: g_constraints.Interval = g_constraints.Interval(1e-6, 1e-4),
        matern_smoothness: float = 2.5,
) -> Tuple[b_models.SingleTaskGP, g_lh.GaussianLikelihood]:
    likelihood: g_lh.GaussianLikelihood = g_lh.GaussianLikelihood(
        noise_constraint=noise_interval,
    )
    kernel: g_kern.ScaleKernel = g_kern.ScaleKernel(
        base_kernel=g_kern.MaternKernel(
            nu=matern_smoothness,
        )
    )
    model: b_models.SingleTaskGP = b_models.SingleTaskGP(
        train_X=X,
        train_Y=Y,
        likelihood=likelihood,
        covar_module=kernel,
    )
    mll: g_mlls.ExactMarginalLogLikelihood = g_mlls.ExactMarginalLogLikelihood(
        likelihood=likelihood,
        model=model
    )
    return (model, mll)

def sample_candidate_positions(
    dim: int,
    batch_size: int,
    scramble: Optional[bool] = True, 
) -> Tensor:
    sobol: t_qrand.SobolEngine = t_qrand.SobolEngine(
        dimension=dim,
        scramble=scramble,
        
    )
    raw_samples: Tensor = sobol.draw(n=batch_size).to(device=device, dtype=tensor_dtype).requires_grad_(True)
    return (lb + (ub - lb) * raw_samples).requires_grad_(True)

def get_ucb(
        model: b_models.SingleTaskGP,
        X: Tensor,
    ) -> Tensor:
    posterior: bp_gpytorch.GPyTorchPosterior = model.posterior(X=X)
    return posterior.mean + confidence * torch.sqrt(posterior.variance)

def get_lcb(
        model: b_models.SingleTaskGP, 
        X: Tensor
        ) -> Tensor:
    posterior: bp_gpytorch.GPyTorchPosterior = model.posterior(X=X)
    return posterior.mean - confidence * torch.sqrt(posterior.variance)

def pessimistic_safe_subset(
        model: b_models.SingleTaskGP,
        X: Tensor,
    ) -> Tensor:
    ucb_tensor: Tensor = get_lcb(
        model=model,
        X=X,
    ).squeeze(1)
    return X[torch.argmin(ucb_tensor, dim=0)]

def optimistic_safe_subset(
        model: b_models.SingleTaskGP,
        Z: Tensor,
        X: Tensor,
    ) -> Tensor:

    X_ucb_tensor: Tensor = get_ucb(
        model=model,
        X=X,
    )

    posterior_X: bp_gpytorch.GPyTorchPosterior = model.posterior(X=X)
    mean_flat: Tensor = posterior_X.mean.flatten()

    gradients: Tensor = torch.linalg.norm(
        torch.autograd.grad(
            outputs=mean_flat,
            inputs=X,
            grad_outputs=torch.ones_like(mean_flat),
        )[0],
        ord=float("inf"),
        dim=1
    )

    eucl_distance_XZ: Tensor = torch.cdist(
        x1=X,
        x2=Z,
    )


    L_i: Tensor = torch.amax(gradients)
    safety: Tensor = X_ucb_tensor - L_i * eucl_distance_XZ
    mask: Tensor = (safety >= 0.0).any(dim=0)

    Z_lcb_tensor: Tensor = get_lcb(
        model=model,
        X=Z,
    ).squeeze(1)

    z_masked: Tensor = torch.where(mask, Z_lcb_tensor, float("inf"))
    return Z[torch.argmin(z_masked, dim=0)]

def selection(
        model: b_models.SingleTaskGP,

        X_select: Tensor,
        Z_select: Tensor,

        X: Tensor,
    ) -> Tensor:
    x_lcb: Tensor = get_lcb(model=model, X=X_select.unsqueeze(0))
    z_lcb: Tensor = get_lcb(model=model, X=Z_select.unsqueeze(0))

    if x_lcb.item() < z_lcb.item():
        return X_select
    
    eucl_distance: Tensor = torch.cdist(
        x1=X,
        x2=Z_select.unsqueeze(0)
    )
    return X[torch.argmin(input=eucl_distance, dim=0)]


In [432]:
x_safe: Tensor = torch.rand(
    size=[num_samples, dim],
    requires_grad=True
).to(device=device, dtype=tensor_dtype)
y_safe: Tensor = torch.tensor(
    [eval_ackley(x_safe[i, :]) for i in range(num_samples)],
).to(device=device, dtype=tensor_dtype).unsqueeze(-1)

state: BOState = BOState(
    max_iterations=max_iterations,
)

while not state.is_exceeded():
    state.increment()
    model, mll = create_surrogate(
        X=x_safe.detach(),
        Y=y_safe.detach(),
    )
    
    with gpytorch.settings.max_cholesky_size(max_cholesky_size):
        b_fit.fit_gpytorch_mll(mll=mll)

        z_candidates: Tensor = sample_candidate_positions(
            dim=dim,
            batch_size=batch_size,
            scramble=True
        )

        pessimistic: Tensor = pessimistic_safe_subset(
            model=model,
            X=x_safe,
        )
        optimistic: Tensor = optimistic_safe_subset(
            model=model,
            X=x_safe,
            Z=z_candidates,
        )
        print(pessimistic.shape, optimistic.squeeze(0).shape)
        next_candidates: Tensor = selection(
            model=model,

            X_select=pessimistic,
            Z_select=optimistic.squeeze(0),
            
            X=x_safe,
        )

        y_next: Tensor = torch.tensor(
            [eval_ackley(x) for x in next_candidates], 
            dtype=tensor_dtype, 
            device=device
        ).unsqueeze(-1)

        x_safe: Tensor = torch.cat(
            (x_safe, next_candidates),
            dim=0,
        )
        y_safe: Tensor = torch.cat(
            (y_safe, y_next), 
            dim=0,
        )

        print(f"[{len(x_safe)}]: minimum y value: {torch.amin(y_safe)}")



Current iteration: 1


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78965/2300105273.py:15: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(


torch.Size([25]) torch.Size([25])
[6]: minimum y value: 3.4716577529907227
Current iteration: 2


/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78965/2300105273.py:15: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/meta-pytorch/botorch/discussions/1444
  model: b_models.SingleTaskGP = b_models.SingleTaskGP(
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/gpytorch/distributions/multivariate_normal.py:375: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
/var/folders/95/n78ktpt12l71xwnt2mjjtd0w0000gn/T/ipykernel_78965/2300105273.py:15: InputDataWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/met

torch.Size([25]) torch.Size([25])
[7]: minimum y value: 3.4716577529907227
Current iteration: 3


/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/botorch/fit.py:241: OptimizationWarning: `scipy_minimize` terminated with status OptimizationStatus.FAILURE, displaying original message from `scipy.optimize.minimize`: ABNORMAL: 
  result = optimizer(mll, closure=closure, **optimizer_kwargs)
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/botorch/fit.py:241: OptimizationWarning: `scipy_minimize` terminated with status OptimizationStatus.FAILURE, displaying original message from `scipy.optimize.minimize`: ABNORMAL: 
  result = optimizer(mll, closure=closure, **optimizer_kwargs)
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/botorch/fit.py:241: OptimizationWarning: `scipy_minimize` terminated with status OptimizationStatus.FAILURE, displaying original message from `scipy.optimize.minimize`: ABNORMAL: 
  result = optimizer(mll, closure=closure, **optimizer_kwargs)
/Users/apple/miniconda3/envs/botorch/lib/python3.14/site-packages/botorch

ModelFittingError: All attempts to fit the model have failed.

In [ ]:
import plotly.graph_objects as go
import numpy as np
import torch

def plot_safeopt_objective_plotly(y_safe: torch.Tensor, num_initial_samples: int):
    # 1. Extract only the objective values evaluated DURING the loop
    # We flatten the array so Plotly can read it easily as a 1D list
    y_evaluated = y_safe[num_initial_samples:].detach().cpu().numpy().flatten()
    iterations = np.arange(1, len(y_evaluated) + 1)

    # 2. Initialize the Plotly figure
    fig = go.Figure()

    # 3. Add the SafeOpt objective line (expect high fluctuations!)
    fig.add_trace(
        go.Scatter(
            x=iterations, 
            y=y_evaluated, 
            mode='lines', 
            name='SafeOpt',
            line=dict(color='#1f77b4')
        )
    )

    # 4. Add the dashed red line for the 'minimum value' benchmark (0.0 for Ackley)
    fig.add_trace(
        go.Scatter(
            x=[iterations, iterations[-1]], 
            y=[0.0, 0.0], 
            mode='lines', 
            name='minimum value',
            line=dict(color='firebrick', dash='dash')
        )
    )

    # 5. Format the layout to match the paper's style
    fig.update_layout(
        title='GoOSE Performance (Ackley Function)',
        xaxis_title='Iteration',
        yaxis_title='Objective',
        template='plotly_white', # Gives a clean background similar to matplotlib
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99
        )
    )
    
    fig.show(renderer="vscode") 

plot_safeopt_objective_plotly(
    y_safe=y_safe,
    num_initial_samples=num_samples
)